# 01 - EDA y preparación de datos de Olist

Objetivo: cargar los CSV, convertir fechas, construir `delivery_time_days` y `late_delivery`, agregar a nivel de pedido y revisar composición, faltantes y distribuciones. La predicción se define en el momento de la compra; las señales futuras no serán predictores.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import (
    LEAKAGE_COLUMNS,
    add_targets,
    build_order_level_dataset,
    load_olist_tables,
    predictor_columns,
)

DATA_DIR = ROOT / 'data' / 'raw'
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

## 1. Carga y composición de las tablas

Si falta algún CSV requerido, el cargador muestra un error con la ruta y los nombres esperados. Consulte `data/README.md` para la descarga.

In [ ]:
tables = load_olist_tables(DATA_DIR)
composition = pd.DataFrame(
    {
        'rows': {name: len(frame) for name, frame in tables.items()},
        'columns': {name: frame.shape[1] for name, frame in tables.items()},
        'duplicate_rows': {name: frame.duplicated().sum() for name, frame in tables.items()},
    }
).sort_index()
composition

## 2. Fechas y construcción de objetivos

Los targets usan fechas posteriores únicamente como etiquetas. Una duración negativa se marca como faltante.

In [ ]:
orders_with_targets = add_targets(tables['orders'])
orders_with_targets[[
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'delivery_time_days',
    'late_delivery',
]].head()

In [ ]:
target_summary = pd.DataFrame({
    'non_null': orders_with_targets[['delivery_time_days', 'late_delivery']].notna().sum(),
    'missing': orders_with_targets[['delivery_time_days', 'late_delivery']].isna().sum(),
})
target_summary

## 3. Tabla analítica: una fila por pedido

In [ ]:
orders = build_order_level_dataset(tables)
assert orders['order_id'].is_unique
print(f'Pedidos: {len(orders):,}')
print(f'Variables: {orders.shape[1]}')
orders.head()

## 4. Faltantes y distribuciones

Las decisiones de imputación se implementan dentro del pipeline del notebook 02 para aprenderlas solo con entrenamiento.

In [ ]:
missing = (
    orders.isna().mean().mul(100).rename('missing_pct').to_frame()
    .assign(missing_count=orders.isna().sum())
    .query('missing_count > 0')
    .sort_values('missing_pct', ascending=False)
)
missing.head(25)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(orders['delivery_time_days'].dropna(), bins=50, ax=axes[0])
axes[0].set(title='Tiempo real de entrega', xlabel='Días', ylabel='Pedidos')
late_counts = orders['late_delivery'].value_counts(dropna=False).sort_index()
late_counts.plot.bar(ax=axes[1])
axes[1].set(title='Distribución de retraso', xlabel='late_delivery', ylabel='Pedidos')
plt.tight_layout();

In [ ]:
numeric_for_summary = [
    'delivery_time_days', 'estimated_delivery_days', 'total_price',
    'total_freight_value', 'item_count', 'mean_product_weight_g'
]
orders[numeric_for_summary].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

## 5. Auditoría de variables permitidas

Los identificadores y las señales posteriores a la compra se excluyen. Las variables categóricas se codificarán con one-hot y las numéricas/categóricas se imputarán dentro del pipeline.

In [ ]:
features = predictor_columns(orders)
assert not set(features).intersection(LEAKAGE_COLUMNS)
feature_types = pd.DataFrame({
    'dtype': orders[features].dtypes.astype(str),
    'unique_values': orders[features].nunique(dropna=True),
    'missing_pct': orders[features].isna().mean().mul(100),
})
feature_types

### Próximos controles

- Documentar la versión/fecha de descarga de los CSV.
- Revisar valores extremos con criterios definidos antes de eliminarlos.
- Comparar periodos y estados para detectar cambio temporal o geográfico.
- Exportar figuras al directorio `report/figures/` solo después de ejecutar y revisar el análisis.